**TFIDF**

*Frequency-inverse document frequency (TFIDF)* is a numerical statistic that is intended to reflect how important a word is to a document in a collection or corpus. It is often used as a feature for text classification.

In [1]:
# loader scripts

import requests
import pandas as pd
import numpy as np

def to_tz(df_, time_col, tz_offset, tz_name):
    return (df_
    .groupby(tz_offset)
    [time_col]
    .transform(lambda s: pd.to_datetime(s)
    .dt.tz_localize(s.name, ambiguous=True)
    .dt.tz_convert(tz_name))
    )



def load_fuel_economy_data() -> pd.DataFrame :
    url = 'https://www.fueleconomy.gov/feg/epadata/vehicles.csv'
    cols = ['year', 'make', 'model', 'trany', 'drive', 'VClass', 'eng_dscr',
            'barrels08', 'city08', 'comb08', 'range', 'evMotor', 'cylinders', 'displ', 'fuelCost08',
            'fuelType', 'highway08', 'trans_dscr','createdOn']

    raw =  pd.read_csv(url)
    autos = (raw
        .loc[:, cols]
        .assign(
            # Extract Timezone Offset (e.g., EDT -> EST5EDT)
            offset=(raw.createdOn.str.extract(r'\d\d:\d\d (?P<offset>[A-Z]{3}?)')
                    ['offset'].replace('EDT', 'EST5EDT')),

            # Reconstruct date string (removing the original TZ name for parsing)
            str_date=(raw.createdOn.str.slice(4, 19) + " " +
                    raw.createdOn.str.slice(-4)),

            # Convert to localized datetime
            createdOn=lambda df_: to_tz(df_, 'str_date', 'offset', 'America/New_York')
        )
        .drop(columns=['offset', 'str_date']) # Clean up temp columns
        )
    return autos


In [3]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

C:\Users\dsgou\AppData\Local\Temp\ipykernel_27088\501929946.py:24: DtypeWarning: Columns (69,71,72,73,74,75,77,80) have mixed types. Specify dtype option on import or set low_memory=False.
  raw =  pd.read_csv(url)


,year,make,model,trany,drive,VClass,eng_dscr,barrels08,city08,comb08,range,evMotor,cylinders,displ,fuelCost08,fuelType,highway08,trans_dscr,createdOn
0,1985,Alfa Romeo,Spider Veloce 2000,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(FFS),14.167143,19,21,0,NaN,4.0,2.0,2850,Regular,25,NaN,2013-01-01 00:00:00-05:00
1,1985,Ferrari,Testarossa,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(GUZZLER),27.046364,9,11,0,NaN,12.0,4.9,5450,Regular,14,NaN,2013-01-01 00:00:00-05:00
2,1985,Dodge,Charger,Manual 5-spd,Front-Wheel Drive,Subcompact Cars,(FFS),11.018889,23,27,0,NaN,4.0,2.2,2200,Regular,33,SIL,2013-01-01 00:00:00-05:00
3,1985,Dodge,B150/B250 Wagon 2WD,Automatic 3-spd,Rear-Wheel Drive,Vans,NaN,27.046364,10,11,0,NaN,8.0,5.2,5450,Regular,12,NaN,2013-01-01 00:00:00-05:00
4,1993,Subaru,Legacy AWD Turbo,Manual 5-spd,4-Wheel or All-Wheel Drive,Compact Cars,"(FFS,TRBO)",15.658421,17,19,0,NaN,4.0,2.2,3650,Premium,23,NaN,2013-01-01 00:00:00-05:00


In [4]:
# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

In [23]:

# convert text pipeline to sklearn transformer so we can keep in pandas
# and preserve index
from sklearn.base import BaseEstimator, TransformerMixin


def combine_str_cols_transformer(X, cols, new_col_name):
    # tdidf expects a single column of strings
    return X.assign(**{new_col_name: X[cols].fillna('').agg(' '.join, axis='columns')})[new_col_name]


class TextPipeline(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols):
        self.cat_cols = cat_cols
        self.text_pipeline = Pipeline([
            ('combine_str', FunctionTransformer(combine_str_cols_transformer,
                                                kw_args={'cols': cat_cols, 'new_col_name': 'all_str'})),
            ('tfidf', TfidfVectorizer()), # can't be sparse because of Pandas
            ('make_dense', FunctionTransformer(lambda X: X.toarray())),
            ('pca', PCA(n_components=10)),
        ])

    def fit(self, X, y=None):
        self.text_pipeline.fit(X, y)
        return self

    def transform(self, X):
        res = self.text_pipeline.transform(X)
        # replace index with X index
        df = (res
              .assign(index=X.index)
              .set_index('index')
        )
        return df

In [25]:

# Add TFIDF to combination of string columns
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
# replace hashing encoder with target encoder for high cardinality columns

from category_encoders import hashing
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler, OneHotEncoder, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer


# create X and y
X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

# split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# create pipeline fill cylinders with 0 and displ with median

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
displ_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore')
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True)
target_encoder = TargetEncoder(target_type='continuous', random_state=42)

def debug_transformer(X, name):
    globals()[name] = X
    return X

cat_cols =  ['make', 'model', 'trany', 'drive',
            'VClass', 'eng_dscr', 'evMotor', 'fuelType', 'trans_dscr', ]
low_cardinality_cols = ['VClass', 'drive', 'fuelType', 'trany']
high_cardinality_cols = ['make', 'model', 'eng_dscr', 'evMotor', 'trans_dscr']

# Create the column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', displ_imputer, ['displ']),
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        #('hashing_encoder', hashing_encoder, high_cardinality_cols)
        ('target_encoder', target_encoder, high_cardinality_cols),
        ('text', TextPipeline(cat_cols), cat_cols)
    ],
    remainder='passthrough'
)

# pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name': 'tmp_X'})),
    ('std_scaler', std_scaler),
    #('pca', PCA(n_components=10)),
    #  ('minmax_scaler', minmax_scaler, ['range']),
    ('lr', LinearRegression())])

# fit the pipeline
pipeline.fit(X_train, y_train)

AttributeError: 'numpy.ndarray' object has no attribute 'assign'